In [1]:
import csv
import os
import math
import time
from copy import deepcopy

from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid, save_image


In [2]:
@dataclass
class UNetConfig:
    """Small config (~38M params) for single-GPU / RTX Ada 6000."""
    in_ch: int = 3
    out_ch: int = 3
    base_ch: int = 128
    ch_mult: Tuple[int, ...] = (1, 2, 2, 2)
    num_res_blocks: int = 2
    attn_resolutions: Tuple[int, ...] = (16,)
    dropout: float = 0.1
    num_heads: int = 4
    # image_size: int = 32
    # num_classes: int = 10

@dataclass
class MultiResDriftConfig:
    """Single-GPU conditional DriftXpress configuration for CIFAR-10."""

    # Training. Keep 64 negatives per active class because repulsion quality
    # depends strongly on the number of same-class generated samples.
    num_classes: int = 10
    classes_per_step: int = 2
    samples_per_class: int = 128
    batch_size: int = 256

    lr: float = 2e-4
    min_lr: float = 2e-5
    warmup_steps: int = 2_000
    ema_decay: float = 0.9999
    max_grad_norm: float = 2.0
    total_steps: int = 50_000
    temperatures: List[float] = field(
        default_factory=lambda: [0.02, 0.05, 0.2]
    )
    dataset: str = "cifar10"

    # Multi-depth DINO feature encoder.
    encoder: str = "dinov3"
    pool_size: int = 4

    # Conditional Nyström attraction. Each class owns an independent cache,
    # avoiding a single (num_classes * landmarks)^2 landmark matrix.
    nystrom_landmarks_per_class: int = 328

    # Smaller global cache used only by drifting-native CFG. This keeps CFG
    # materially cheaper than using all 328 landmarks from every class.
    nystrom_uncond_landmarks_per_class: int = 64

    nystrom_ridge: float = 1e-4
    nystrom_landmark_seed: int = 0

    # Drifting-native CFG.
    cfg_enabled: bool = True
    cfg_alpha_max: float = 4.0
    cfg_power: float = 3.0
    cfg_no_guidance_probability: float = 0.5
    eval_cfg_scale: float = 1.0

    # Control-variate Nyström attraction.
    cv_enabled: bool = True

    # Number of paired exact/Nyström real features per active class.
    # Set to zero for the exact baseline path.
    cv_samples_per_class: int = 32

    # Precomputed random reservoir per class. This avoids recomputing
    # phi(y) against all landmarks during every training iteration.
    cv_pool_per_class: int = 256

    # "moments" is the recommended estimator.
    # "ratio" implements the originally proposed formula directly.
    cv_mode: str = "moments" # ['moments', 'ratio']

    # beta=1 gives full bias correction. beta<1 shrinks a noisy correction.
    cv_strength: float = 0.5

    # Fall back to the full Nyström barycenter when the corrected
    # denominator is too small relative to the original denominator.
    cv_den_floor_ratio: float = 0.05

    # Store points and precomputed Nyström features in FP16.
    cv_cache_dtype: torch.dtype = torch.float16

    # Independent seed from landmark selection.
    cv_seed: int = 1729

    # Since real features are cached once, storing both orientations gives a
    # genuine flip-augmented reference distribution.
    use_horizontal_flip: bool = True
    feature_cache_dtype: torch.dtype = torch.float16

    # Logging/checkpointing.
    log_every: int = 100
    sample_every: int = 5_000
    save_every: int = 10_000


In [3]:
def timestep_embedding(t, dim, max_period=10000):
    """Sinusoidal timestep embedding. t: [B] long or float -> [B, dim]."""
    half = dim // 2
    freqs = torch.exp(
        -math.log(max_period) * torch.arange(half, device=t.device, dtype=torch.float32) / half
    )
    args = t[:, None].float() * freqs[None]
    embedding = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
    if dim % 2:
        embedding = torch.cat([embedding, torch.zeros_like(embedding[:, :1])], dim=-1)
    return embedding

class GroupNorm32(nn.GroupNorm):
    """GroupNorm that converts to float32 for numerical stability."""
    def forward(self, x):
        return super().forward(x.float()).to(x.dtype)

class ResBlock(nn.Module):
    """ResBlock with timestep conditioning via scale+shift after second GroupNorm."""

    def __init__(self, in_ch, out_ch, time_dim, dropout=0.1):
        super().__init__()
        self.norm1 = GroupNorm32(32, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.time_proj = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_dim, out_ch * 2),
        )
        self.norm2 = GroupNorm32(32, out_ch)
        self.dropout = nn.Dropout(dropout)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

        # Zero-init last conv for residual-friendly start
        nn.init.zeros_(self.conv2.weight)
        nn.init.zeros_(self.conv2.bias)

    def forward(self, x, temb):
        h = F.silu(self.norm1(x))
        h = self.conv1(h)

        # Time conditioning: scale + shift on second norm
        t_out = self.time_proj(temb)[:, :, None, None]
        scale, shift = t_out.chunk(2, dim=1)

        h = self.norm2(h) * (1 + scale) + shift
        h = F.silu(h)
        h = self.dropout(h)
        h = self.conv2(h)

        return h + self.skip(x)

class SelfAttention(nn.Module):
    """Multi-head self-attention with pre-norm."""

    def __init__(self, ch, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.norm = GroupNorm32(32, ch)
        self.qkv = nn.Conv1d(ch, ch * 3, 1)
        self.out = nn.Conv1d(ch, ch, 1)
        nn.init.zeros_(self.out.weight)
        nn.init.zeros_(self.out.bias)

    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x).reshape(B, C, H * W)
        qkv = self.qkv(h).reshape(B, 3, self.num_heads, C // self.num_heads, H * W)
        q, k, v = qkv.unbind(1)  # each [B, heads, head_dim, HW]
        q = q.permute(0, 1, 3, 2)  # [B, heads, HW, head_dim]
        k = k.permute(0, 1, 3, 2)
        v = v.permute(0, 1, 3, 2)
        h = F.scaled_dot_product_attention(q, k, v)
        h = h.permute(0, 1, 3, 2).reshape(B, C, H * W)
        h = self.out(h).reshape(B, C, H, W)
        return x + h

class ResAttnBlock(nn.Module):
    """ResBlock + optional SelfAttention."""

    def __init__(self, in_ch, out_ch, time_dim, dropout, use_attn, num_heads):
        super().__init__()
        self.res = ResBlock(in_ch, out_ch, time_dim, dropout)
        self.attn = SelfAttention(out_ch, num_heads) if use_attn else None

    def forward(self, x, temb):
        x = self.res(x, temb)
        if self.attn is not None:
            x = self.attn(x)
        return x

class Downsample(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv = nn.Conv2d(ch, ch, 3, stride=2, padding=1)

    def forward(self, x):
        return self.conv(x)

class Upsample(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv = nn.Conv2d(ch, ch, 3, padding=1)

    def forward(self, x):
        x = F.interpolate(x, scale_factor=2, mode="nearest")
        return self.conv(x)

class UNet(nn.Module):
    """
    Standard UNet for 32x32 images.

    Architecture:
    - Channel progression: base_ch * ch_mult at each resolution level
    - Spatial: 32 -> 16 -> 8 -> 4 (with 3 downsamples)
    - Self-attention at specified resolutions
    - Timestep conditioning via AdaGN (scale+shift)
    - Skip connections between encoder and decoder
    """

    def __init__(
        self,
        in_ch=3,
        out_ch=3,
        base_ch=128,
        ch_mult=(1, 2, 2, 2),
        num_res_blocks=2,
        attn_resolutions=(16,),
        dropout=0.1,
        num_heads=4,
        num_classes=0,
        image_size=32,
    ):
        super().__init__()
        self.in_ch = in_ch
        self.out_ch = out_ch
        self.image_size = image_size
        self.base_ch = base_ch
        self.ch_mult = ch_mult
        self.num_res_blocks = num_res_blocks
        self.num_classes = num_classes
        time_dim = base_ch * 4

        # Time embedding MLP
        self.time_embed = nn.Sequential(
            nn.Linear(base_ch, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim),
        )

        # Class conditioning (added to time embedding)
        if num_classes > 0:
            self.class_embed = nn.Embedding(num_classes, time_dim)
            self.cfg_embed = nn.Sequential(
                nn.Linear(base_ch, time_dim),
                nn.SiLU(),
                nn.Linear(time_dim, time_dim),
            )

        # Input conv
        self.conv_in = nn.Conv2d(in_ch, base_ch, 3, padding=1)

        # --- Down path ---
        self.down_blocks = nn.ModuleList()
        self.down_samples = nn.ModuleList()
        skip_chs = [base_ch]
        ch = base_ch
        res = image_size

        for level, mult in enumerate(ch_mult):
            out = base_ch * mult
            use_attn = res in attn_resolutions
            for _ in range(num_res_blocks):
                self.down_blocks.append(
                    ResAttnBlock(ch, out, time_dim, dropout, use_attn, num_heads)
                )
                ch = out
                skip_chs.append(ch)

            if level < len(ch_mult) - 1:
                self.down_samples.append(Downsample(ch))
                skip_chs.append(ch)
                res //= 2

        # --- Middle ---
        self.mid_block1 = ResAttnBlock(ch, ch, time_dim, dropout, True, num_heads)
        self.mid_block2 = ResAttnBlock(ch, ch, time_dim, dropout, False, num_heads)

        # --- Up path ---
        self.up_blocks = nn.ModuleList()
        self.up_samples = nn.ModuleList()

        for level in reversed(range(len(ch_mult))):
            mult = ch_mult[level]
            out = base_ch * mult
            use_attn = res in attn_resolutions
            for _ in range(num_res_blocks + 1):
                skip_ch = skip_chs.pop()
                self.up_blocks.append(
                    ResAttnBlock(ch + skip_ch, out, time_dim, dropout, use_attn, num_heads)
                )
                ch = out

            if level > 0:
                self.up_samples.append(Upsample(ch))
                res *= 2

        # Output
        self.norm_out = GroupNorm32(32, ch)
        self.conv_out = nn.Conv2d(ch, out_ch, 3, padding=1)
        nn.init.zeros_(self.conv_out.weight)
        nn.init.zeros_(self.conv_out.bias)

    def forward(self, x, t=None, class_labels=None, cfg_scale=None):
        """
        Args:
            x: [B, C, H, W] input image (noisy for DDPM, noise for drift)
            t: [B] integer timesteps. If None, uses t=0 (for drift model).
            class_labels: [B] integer class labels. Only used when num_classes > 0.
            cfg_scale: [B] or scalar CFG scale. Used by conditional OT runs.

        Returns:
            [B, C, H, W] predicted noise (DDPM) or generated image (drift)
        """
        B = x.shape[0]
        if t is None:
            t = torch.zeros(B, device=x.device, dtype=torch.long)

        # Time embedding
        temb = timestep_embedding(t, self.base_ch)
        temb = self.time_embed(temb)

        # Class conditioning
        if self.num_classes > 0 and class_labels is not None:
            temb = temb + self.class_embed(class_labels)
        if self.num_classes > 0 and cfg_scale is not None:
            cfg_scale = torch.as_tensor(cfg_scale, device=x.device, dtype=torch.float32)
            if cfg_scale.ndim == 0:
                cfg_scale = cfg_scale.repeat(B)
            cfg_temb = self.cfg_embed(timestep_embedding(cfg_scale, self.base_ch))
            temb = temb + 0.02 * cfg_temb

        # Input
        h = self.conv_in(x)
        skips = [h]

        # Down
        block_idx = 0
        ds_idx = 0
        for level in range(len(self.ch_mult)):
            for _ in range(self.num_res_blocks):
                h = self.down_blocks[block_idx](h, temb)
                skips.append(h)
                block_idx += 1
            if level < len(self.ch_mult) - 1:
                h = self.down_samples[ds_idx](h)
                skips.append(h)
                ds_idx += 1

        # Middle
        h = self.mid_block1(h, temb)
        h = self.mid_block2(h, temb)

        # Up
        block_idx = 0
        us_idx = 0
        for level in reversed(range(len(self.ch_mult))):
            for _ in range(self.num_res_blocks + 1):
                h = torch.cat([h, skips.pop()], dim=1)
                h = self.up_blocks[block_idx](h, temb)
                block_idx += 1
            if level > 0:
                h = self.up_samples[us_idx](h)
                us_idx += 1

        # Output
        h = F.silu(self.norm_out(h))
        h = self.conv_out(h)
        return h

class EMA:
    """Exponential Moving Average of model parameters."""

    def __init__(self, model: nn.Module, decay: float = 0.9999):
        self.decay = decay
        self.shadow = deepcopy(model)
        self.shadow.eval()
        self.shadow.requires_grad_(False)

    @torch.no_grad()
    def update(self, model: nn.Module):
        for s_param, m_param in zip(self.shadow.parameters(), model.parameters()):
            s_param.mul_(self.decay).add_(m_param.data, alpha=1 - self.decay)
        for s_buf, m_buf in zip(self.shadow.buffers(), model.buffers()):
            s_buf.copy_(m_buf)

    def forward(self, *args, **kwargs):
        return self.shadow(*args, **kwargs)

    def state_dict(self):
        return self.shadow.state_dict()

    def load_state_dict(self, state_dict):
        self.shadow.load_state_dict(state_dict)


In [4]:
# --- Original: methods/driftxpress.py: unsharded Nyström path ----------------
@dataclass
class NystromStats:
    """Cached Nyström summaries for one feature group and one temperature."""

    landmarks: torch.Tensor
    A: torch.Tensor
    global_totals: torch.Tensor
    global_weighted_points: torch.Tensor
    temperature: float
    num_points: int


def move_nystrom_stats_to_device(
    stats: NystromStats,
    device: torch.device | str,
) -> NystromStats:
    """Move a Nyström cache block to the requested device if needed."""
    device = torch.device(device)

    if (
        stats.landmarks.device == device
        and stats.A.device == device
        and stats.global_totals.device == device
        and stats.global_weighted_points.device == device
    ):
        return stats

    return NystromStats(
        landmarks=stats.landmarks.to(
            device=device, dtype=torch.float32, non_blocking=True
        ),
        A=stats.A.to(
            device=device, dtype=torch.float32, non_blocking=True
        ),
        global_totals=stats.global_totals.to(
            device=device, dtype=torch.float32, non_blocking=True
        ),
        global_weighted_points=stats.global_weighted_points.to(
            device=device, dtype=torch.float32, non_blocking=True
        ),
        temperature=float(stats.temperature),
        num_points=int(stats.num_points),
    )


def normalize_drift_batched(V, D=None, eps=1e-8):
    """Normalize drift per location so E[||V||^2 / D] ~ 1."""
    if D is None:
        D = V.shape[-1]
    lambda_j = torch.sqrt((V.float().pow(2).sum(dim=-1) / D).mean(dim=-1)).detach()
    return V / (lambda_j.unsqueeze(-1).unsqueeze(-1) + eps)


def _laplacian_kernel_batched(x, y, temperature, eps=1e-8):
    """Batched Laplacian kernel."""
    D = x.shape[-1]
    tau_tilde = max(float(temperature) * float(D), eps)
    dist = torch.cdist(x.float(), y.float(), p=2)
    return torch.exp(-dist / tau_tilde)


def _inverse_sqrt_psd_batched(mats, eps=1e-8):
    """Compute batched inverse square root for symmetric PSD matrices."""
    eigvals, eigvecs = torch.linalg.eigh(mats.float())
    inv_sqrt_eigs = eigvals.clamp_min(eps).rsqrt()
    return (eigvecs * inv_sqrt_eigs.unsqueeze(-2)) @ eigvecs.transpose(-1, -2)


def _compute_exact_repulsive_barycenter_batched(x, temperature=0.05, eps=1e-8, mask_self=True):
    """Compute an exact kernel repulsive barycenter over the current batch."""
    x_f = x.float()
    D = x.shape[-1]
    tau_tilde = max(float(temperature) * float(D), eps)
    dist = torch.cdist(x_f, x_f, p=2)

    if mask_self:
        N = x.shape[1]
        dist = dist + torch.eye(N, device=x.device, dtype=dist.dtype).unsqueeze(0) * 1e6

    weights = torch.exp(-dist / tau_tilde)
    den = weights.sum(dim=2, keepdim=True).clamp_min(eps)
    return torch.bmm(weights, x_f) / den


def prepare_nystrom_landmarks_batched(
    public_landmarks,
    temperature=0.05,
    ridge=1e-4,
    eps=1e-8,
    device=None,
):
    """Prepare landmark tensors and A = (W + ridge I)^(-1/2)."""
    if public_landmarks.ndim != 3:
        raise ValueError("Expected public_landmarks with shape [M, L, D].")

    if device is None:
        device = public_landmarks.device

    landmarks_t = public_landmarks.transpose(0, 1).contiguous().to(device=device, dtype=torch.float32)
    L, M, _ = landmarks_t.shape
    W = _laplacian_kernel_batched(landmarks_t, landmarks_t, temperature=temperature, eps=eps)
    eye = torch.eye(M, device=device, dtype=W.dtype).unsqueeze(0).expand(L, -1, -1)
    A = _inverse_sqrt_psd_batched(W + ridge * eye, eps=eps)
    return landmarks_t, A


def compute_nystrom_features_batched(x, landmarks, A, temperature=0.05, eps=1e-8):
    """Explicit Nyström features phi(x) = K(x, U) (W + ridge I)^(-1/2)."""
    K_xu = _laplacian_kernel_batched(x, landmarks, temperature=temperature, eps=eps)
    return torch.bmm(K_xu, A)


def _compute_positive_terms_batched(x, stats, eps=1e-8):
    """Compute attractive Nyström numerator and denominator terms."""
    stats = move_nystrom_stats_to_device(stats, x.device)
    phi = compute_nystrom_features_batched(
        x=x.float(),
        landmarks=stats.landmarks,
        A=stats.A,
        temperature=stats.temperature,
        eps=eps,
    )
    pos_num = torch.bmm(phi, stats.global_weighted_points.float())
    pos_den = torch.sum(
        phi * stats.global_totals.unsqueeze(1).float(),
        dim=-1,
        keepdim=True,
    ).clamp_min(eps)
    return phi, pos_num, pos_den

def _compute_nystrom_moments_batched(x, stats, eps=1e-8):
    """Return Nyström numerator and denominator before barycenter division.

    Raw sums are returned because CFG must combine distributions before
    dividing by the combined kernel mass.
    """
    stats = move_nystrom_stats_to_device(stats, x.device)

    phi = compute_nystrom_features_batched(
        x=x.float(),
        landmarks=stats.landmarks,
        A=stats.A,
        temperature=stats.temperature,
        eps=eps,
    )

    numerator = torch.bmm(
        phi,
        stats.global_weighted_points.float(),
    )
    denominator = torch.sum(
        phi * stats.global_totals.unsqueeze(1).float(),
        dim=-1,
        keepdim=True,
    )

    return numerator, denominator


def _compute_exact_repulsive_moments_batched(x, temperature=0.05, eps=1e-8):
    """Exact leave-one-out moments of the current generated distribution.

    Returns empirical expectations rather than a pre-divided barycenter so
    they can be mixed correctly with unconditional CFG moments.
    """
    x_f = x.float()
    L, N, _ = x_f.shape

    if N < 2:
        raise ValueError(
            "Exact class-conditional repulsion needs at least two "
            "generated samples per active class."
        )

    D = x_f.shape[-1]
    tau_tilde = max(float(temperature) * float(D), eps)

    dist = torch.cdist(x_f, x_f, p=2)
    eye = torch.eye(N, device=x.device, dtype=dist.dtype).unsqueeze(0)
    dist = dist + eye * 1e6

    weights = torch.exp(-dist / tau_tilde)

    # Express both quantities as empirical expectations. The common factor
    # cancels without CFG but matters when mixing q(.|c) with p_uncond.
    support_size = float(N - 1)
    numerator = torch.bmm(weights, x_f) / support_size
    denominator = weights.sum(dim=2, keepdim=True) / support_size

    return numerator, denominator

def compute_nystrom_cfg_drift_batched(
    x,
    positive_stats,
    unconditional_stats,
    cfg_alpha=1.0,
    eps=1e-8,
    max_drift_norm=None,
    correction_points=None,
    correction_phi=None,
    cv_mode="moments",
    cv_strength=1.0,
    cv_den_floor_ratio=0.05,
):
    """Conditional drift with control-variate positive attraction."""
    x_f = x.float()

    b_pos = _compute_cv_positive_barycenter_batched(
        x=x_f,
        stats=positive_stats,
        correction_points=correction_points,
        correction_phi=correction_phi,
        cv_mode=cv_mode,
        cv_strength=cv_strength,
        cv_den_floor_ratio=cv_den_floor_ratio,
        eps=eps,
    )

    q_num, q_den = _compute_exact_repulsive_moments_batched(
        x_f,
        temperature=positive_stats.temperature,
        eps=eps,
    )

    alpha = max(float(cfg_alpha), 1.0)
    gamma = 1.0 - 1.0 / alpha

    if gamma > 0.0:
        unc_num, unc_den = _compute_nystrom_moments_batched(
            x_f,
            unconditional_stats,
            eps=eps,
        )

        unc_num = unc_num / float(
            unconditional_stats.num_points
        )
        unc_den = unc_den / float(
            unconditional_stats.num_points
        )

        neg_num = (
            (1.0 - gamma) * q_num
            + gamma * unc_num
        )
        neg_den = (
            (1.0 - gamma) * q_den
            + gamma * unc_den
        )
    else:
        neg_num = q_num
        neg_den = q_den

    b_neg = neg_num / neg_den.clamp_min(eps)
    V = (b_pos - b_neg).to(x.dtype)

    if max_drift_norm is not None:
        norms = torch.linalg.vector_norm(
            V.float(),
            dim=-1,
            keepdim=True,
        )

        scale = torch.clamp(
            float(max_drift_norm) / (norms + eps),
            max=1.0,
        )

        V = V * scale.to(V.dtype)

    return V

def compute_nystrom_cfg_drift_multitemp_batched(
    x,
    positive_stats_by_temp,
    unconditional_stats_by_temp,
    cfg_alpha,
    temps=(0.02, 0.05, 0.2),
    eps=1e-8,
    max_drift_norm=None,
    correction_points=None,
    correction_phi_by_temp=None,
    cv_mode="moments",
    cv_strength=1.0,
    cv_den_floor_ratio=0.05,
):
    V_total = torch.zeros_like(x)

    for temp in temps:
        temp = float(temp)

        correction_phi = None
        if correction_phi_by_temp is not None:
            correction_phi = correction_phi_by_temp[temp]

        V_tau = compute_nystrom_cfg_drift_batched(
            x=x,
            positive_stats=positive_stats_by_temp[temp],
            unconditional_stats=unconditional_stats_by_temp[temp],
            cfg_alpha=cfg_alpha,
            eps=eps,
            max_drift_norm=max_drift_norm,
            correction_points=correction_points,
            correction_phi=correction_phi,
            cv_mode=cv_mode,
            cv_strength=cv_strength,
            cv_den_floor_ratio=cv_den_floor_ratio,
        )

        V_total = V_total + normalize_drift_batched(
            V_tau,
            eps=eps,
        )

    return V_total


def drifting_loss_multires_nystrom_conditional(
    gen_groups,
    class_stats_groups,
    unconditional_stats_groups,
    class_labels,
    cfg_scales,
    cv_banks=None,
    cfg=None,
    temps=(0.02, 0.05, 0.2),
    eps=1e-8,
    max_drift_norm=None,
):
    """Class-conditional multi-feature DriftXpress loss with CFG.

    Exact repulsion is evaluated independently within each active class.
    """
    if len(gen_groups) != len(class_stats_groups):
        raise ValueError(
            "gen_groups and class_stats_groups must have the same length."
        )
    if len(gen_groups) != len(unconditional_stats_groups):
        raise ValueError(
            "gen_groups and unconditional_stats_groups must match."
        )

    class_labels = class_labels.to(
        device=gen_groups[0][0].device,
        dtype=torch.long,
    )
    cfg_scales = cfg_scales.to(
        device=gen_groups[0][0].device,
        dtype=torch.float32,
    )

    active_classes = torch.unique(class_labels, sorted=True)
    total_loss = gen_groups[0][0].new_tensor(0.0)
    term_count = 0

    for group_index, (gen_feat, _) in enumerate(gen_groups):
        for class_id_tensor in active_classes:
            class_id = int(class_id_tensor.item())
            indices = torch.nonzero(
                class_labels == class_id_tensor,
                as_tuple=False,
            ).squeeze(1)

            if indices.numel() < 2:
                raise ValueError(
                    f"Class {class_id} has fewer than two generated samples."
                )

            class_feat = gen_feat.index_select(0, indices)
            class_alpha = float(cfg_scales[indices[0]].item())

            with torch.no_grad():
                x = class_feat.detach().transpose(0, 1).contiguous()

                correction_points = None
                correction_phi_by_temp = None

                if (
                    cfg is not None
                    and cfg.cv_enabled
                    and cfg.cv_samples_per_class > 0
                ):
                    if cv_banks is None:
                        raise ValueError(
                            "CV is enabled but cv_banks is None."
                        )

                    bank = cv_banks[group_index][class_id]

                    (
                        correction_points,
                        correction_phi_by_temp,
                    ) = sample_cv_correction_batch(
                        bank=bank,
                        count=cfg.cv_samples_per_class,
                    )

                V = compute_nystrom_cfg_drift_multitemp_batched(
                    x=x,
                    positive_stats_by_temp=(
                        class_stats_groups[group_index][class_id]
                    ),
                    unconditional_stats_by_temp=(
                        unconditional_stats_groups[group_index]
                    ),
                    cfg_alpha=class_alpha,
                    temps=temps,
                    eps=eps,
                    max_drift_norm=max_drift_norm,
                    correction_points=correction_points,
                    correction_phi_by_temp=(
                        correction_phi_by_temp
                    ),
                    cv_mode=(
                        cfg.cv_mode if cfg is not None
                        else "moments"
                    ),
                    cv_strength=(
                        cfg.cv_strength if cfg is not None
                        else 1.0
                    ),
                    cv_den_floor_ratio=(
                        cfg.cv_den_floor_ratio
                        if cfg is not None
                        else 0.05
                    ),
                )

                target = (
                    class_feat.detach()
                    + V.transpose(0, 1)
                )

            total_loss = total_loss + F.mse_loss(
                class_feat,
                target,
            )
            term_count += 1

    return total_loss / max(term_count, 1)

@torch.no_grad()
def precompute_nystrom_statistics_batched(
    sensitive_points,
    public_landmarks,
    temperature=0.05,
    ridge=1e-4,
    eps=1e-8,
    batch_size=512,
    device=None,
):
    """Precompute cached Nyström summaries over private points."""
    if sensitive_points.ndim != 3:
        raise ValueError("Expected sensitive_points with shape [Np, L, D].")
    if public_landmarks.ndim != 3:
        raise ValueError("Expected public_landmarks with shape [M, L, D].")

    if device is None:
        device = sensitive_points.device

    landmarks, A = prepare_nystrom_landmarks_batched(
        public_landmarks=public_landmarks,
        temperature=temperature,
        ridge=ridge,
        eps=eps,
        device=device,
    )

    L, M, D = landmarks.shape
    global_totals = torch.zeros(L, M, device=device, dtype=torch.float32)
    global_weighted_points = torch.zeros(L, M, D, device=device, dtype=torch.float32)

    Np = sensitive_points.shape[0]
    for start in range(0, Np, batch_size):
        end = min(start + batch_size, Np)
        batch = sensitive_points[start:end].to(device=device, dtype=torch.float32, non_blocking=True)
        batch_t = batch.transpose(0, 1).contiguous()
        phi = compute_nystrom_features_batched(
            batch_t,
            landmarks,
            A,
            temperature=temperature,
            eps=eps,
        )
        global_totals += phi.sum(dim=1)
        global_weighted_points += torch.bmm(phi.transpose(1, 2), batch_t)

    return NystromStats(
        landmarks=landmarks,
        A=A,
        global_totals=global_totals,
        global_weighted_points=global_weighted_points,
        temperature=float(temperature),
        num_points=int(Np),
    )


@torch.no_grad()
def precompute_nystrom_statistics_multitemp_batched(
    sensitive_points,
    public_landmarks,
    temps=(0.02, 0.05, 0.2),
    ridge=1e-4,
    eps=1e-8,
    batch_size=512,
    device=None,
):
    """Precompute private summaries for multiple temperatures."""
    stats = {}
    for temp in temps:
        stats[float(temp)] = precompute_nystrom_statistics_batched(
            sensitive_points=sensitive_points,
            public_landmarks=public_landmarks,
            temperature=float(temp),
            ridge=ridge,
            eps=eps,
            batch_size=batch_size,
            device=device,
        )
    return stats

def compute_nystrom_drift_batched(
    x,
    stats,
    eps=1e-8,
    max_drift_norm=None,
    mask_self=True,
):
    """Compute Nyström drift V(x_i) = b_pos(x_i) - b_neg(x_i)."""
    x_f = x.float()
    phi, pos_num, pos_den = _compute_positive_terms_batched(x=x_f, stats=stats, eps=eps)
    b_pos = pos_num / pos_den

    b_neg = _compute_exact_repulsive_barycenter_batched(
        x=x_f,
        temperature=stats.temperature,
        eps=eps,
        mask_self=mask_self,
    )

    V = (b_pos - b_neg).to(x.dtype)

    if max_drift_norm is not None:
        norms = torch.linalg.vector_norm(V.float(), dim=-1, keepdim=True)
        scale = torch.clamp(float(max_drift_norm) / (norms + eps), max=1.0)
        V = V * scale.to(V.dtype)

    return V

def compute_nystrom_drift_multitemp_batched(
    x,
    stats_by_temp,
    temps=(0.02, 0.05, 0.2),
    eps=1e-8,
    max_drift_norm=None,
):
    """Aggregate Nyström drifts over multiple temperatures."""
    V_total = torch.zeros_like(x)
    for temp in temps:
        temp = float(temp)
        if temp not in stats_by_temp:
            raise KeyError(f"Missing Nyström cache for temperature {temp}.")

        V_tau = compute_nystrom_drift_batched(
            x=x,
            stats=stats_by_temp[temp],
            eps=eps,
            max_drift_norm=max_drift_norm,
            mask_self=True
        )
        V_total = V_total + normalize_drift_batched(V_tau, eps=eps)

    return V_total

def drifting_loss_multires_nystrom(
    gen_groups,
    stats_groups,
    temps=(0.02, 0.05, 0.2),
    eps=1e-8,
    max_drift_norm=None
):
    """Multi-resolution drifting loss using Nyström drift."""
    if len(gen_groups) != len(stats_groups):
        raise ValueError("gen_groups and stats_groups must have the same length.")

    total_loss = gen_groups[0][0].new_tensor(0.0)

    for (gen_feat, _), stats_by_temp in zip(gen_groups, stats_groups):
        with torch.no_grad():
            gen_t = gen_feat.detach().transpose(0, 1).contiguous()
            V = compute_nystrom_drift_multitemp_batched(
                x=gen_t,
                stats_by_temp=stats_by_temp,
                temps=temps,
                eps=eps,
                max_drift_norm=max_drift_norm
            )
            target = gen_feat.detach() + V.transpose(0, 1)

        total_loss = total_loss + F.mse_loss(gen_feat, target)

    return total_loss / len(gen_groups)

def _compute_cv_positive_barycenter_batched(
    x,
    stats,
    correction_points=None,
    correction_phi=None,
    cv_mode="moments",
    cv_strength=1.0,
    cv_den_floor_ratio=0.05,
    eps=1e-8,
):
    """Full-data Nyström attraction plus a paired exact correction."""
    stats = move_nystrom_stats_to_device(stats, x.device)

    phi_x = compute_nystrom_features_batched(
        x=x.float(),
        landmarks=stats.landmarks,
        A=stats.A,
        temperature=stats.temperature,
        eps=eps,
    )

    # Cached quantities are raw sums over the full real dataset.
    full_num_sum = torch.bmm(
        phi_x,
        stats.global_weighted_points.float(),
    )

    full_den_sum = torch.sum(
        phi_x * stats.global_totals.unsqueeze(1).float(),
        dim=-1,
        keepdim=True,
    )

    # Convert to empirical expectations before adding mini estimates.
    full_num = full_num_sum / float(stats.num_points)
    full_den = full_den_sum / float(stats.num_points)

    full_barycenter = full_num / full_den.clamp_min(eps)

    if correction_points is None or correction_phi is None:
        return full_barycenter

    exact_num, exact_den = _compute_exact_positive_moments_batched(
        x=x,
        real_points=correction_points,
        temperature=stats.temperature,
        eps=eps,
    )

    mini_nys_num, mini_nys_den = (
        _compute_nystrom_minibatch_moments_batched(
            phi_x=phi_x,
            real_points=correction_points,
            phi_y=correction_phi,
        )
    )

    beta = float(cv_strength)

    if cv_mode == "moments":
        corrected_num = (
            full_num
            + beta * (exact_num - mini_nys_num)
        )
        corrected_den = (
            full_den
            + beta * (exact_den - mini_nys_den)
        )

        # Relative floor prevents a noisy correction from creating an
        # arbitrarily large barycenter when kernel mass is nearly zero.
        den_floor = torch.maximum(
            torch.full_like(corrected_den, eps),
            float(cv_den_floor_ratio)
            * full_den.detach().clamp_min(eps),
        )

        valid = torch.isfinite(corrected_den) & (
            corrected_den > den_floor
        )

        corrected_barycenter = (
            corrected_num
            / corrected_den.clamp_min(eps)
        )

        # Fall back per query/location rather than discarding the complete
        # class correction.
        return torch.where(
            valid.expand_as(corrected_barycenter),
            corrected_barycenter,
            full_barycenter,
        )

    if cv_mode == "ratio":
        exact_barycenter = (
            exact_num / exact_den.clamp_min(eps)
        )

        mini_nys_barycenter = (
            mini_nys_num / mini_nys_den.clamp_min(eps)
        )

        corrected_barycenter = (
            full_barycenter
            + beta
            * (exact_barycenter - mini_nys_barycenter)
        )

        finite = torch.isfinite(corrected_barycenter).all(
            dim=-1,
            keepdim=True,
        )

        return torch.where(
            finite.expand_as(corrected_barycenter),
            corrected_barycenter,
            full_barycenter,
        )

    raise ValueError(f"Unknown cv_mode: {cv_mode}")


In [5]:
@dataclass
class CVCorrectionBank:
    """Cached exact points and their Nyström features for one class/group."""

    # [K, L, D]
    points: torch.Tensor

    # temp -> [L, K, M]
    phi_by_temp: Dict[float, torch.Tensor]

    num_points: int

@torch.no_grad()
def build_cv_correction_banks(
    feature_groups,
    labels,
    class_stats_groups,
    cfg,
    device,
):
    """Build small class-specific control-variate reservoirs.

    This precomputes phi(y) for the reservoir, avoiding an additional
    real-point-to-landmark kernel evaluation on every training step.
    """
    labels = torch.as_tensor(labels, dtype=torch.long, device="cpu")
    banks = []

    for group_index, features in enumerate(feature_groups):
        print(f"Building CV correction bank for group {group_index}...")
        group_banks = {}

        for class_id in range(cfg.num_classes):
            class_indices = torch.nonzero(
                labels == class_id,
                as_tuple=False,
            ).squeeze(1)

            pool_count = min(
                int(cfg.cv_pool_per_class),
                int(class_indices.numel()),
            )

            generator = torch.Generator(device="cpu").manual_seed(
                int(cfg.cv_seed)
                + class_id * 1009
                + group_index * 1_000_003
            )

            permutation = torch.randperm(
                class_indices.numel(),
                generator=generator,
            )

            pool_indices = class_indices.index_select(
                0, permutation[:pool_count]
            )

            points = features.index_select(
                0, pool_indices
            ).to(
                device=device,
                dtype=cfg.cv_cache_dtype,
                non_blocking=True,
            )

            # [L, K, D]
            points_t = points.float().transpose(0, 1).contiguous()

            phi_by_temp = {}

            for temp in cfg.temperatures:
                temp = float(temp)
                stats = class_stats_groups[group_index][class_id][temp]
                stats = move_nystrom_stats_to_device(stats, device)

                phi = compute_nystrom_features_batched(
                    x=points_t,
                    landmarks=stats.landmarks,
                    A=stats.A,
                    temperature=temp,
                )

                phi_by_temp[temp] = phi.to(
                    dtype=cfg.cv_cache_dtype
                )

            group_banks[class_id] = CVCorrectionBank(
                points=points,
                phi_by_temp=phi_by_temp,
                num_points=pool_count,
            )

            point_mb = (
                points.numel() * points.element_size() / 2**20
            )
            phi_mb = sum(
                p.numel() * p.element_size()
                for p in phi_by_temp.values()
            ) / 2**20

            print(
                f"  group={group_index}, class={class_id}: "
                f"pool={pool_count}, points={point_mb:.1f} MiB, "
                f"phi={phi_mb:.1f} MiB"
            )

        banks.append(group_banks)

    return banks

@torch.no_grad()
def sample_cv_correction_batch(
    bank: CVCorrectionBank,
    count: int,
):
    """Sample one paired exact/Nyström correction batch.

    The same indices are used for all temperatures.
    """
    count = min(int(count), int(bank.num_points))

    if count <= 0:
        return None, None

    device = bank.points.device

    indices = torch.randperm(
        bank.num_points,
        device=device,
    )[:count]

    # [L, m, D]
    points_t = bank.points.index_select(
        0, indices
    ).float().transpose(0, 1).contiguous()

    # Each value is [L, m, M].
    phi_by_temp = {
        float(temp): phi.index_select(1, indices).float()
        for temp, phi in bank.phi_by_temp.items()
    }

    return points_t, phi_by_temp

def _compute_exact_positive_moments_batched(
    x,
    real_points,
    temperature,
    eps=1e-8,
):
    """Exact positive moments against a small real-data minibatch.

    Args:
        x: [L, N, D]
        real_points: [L, m, D]

    Returns:
        numerator: [L, N, D]
        denominator: [L, N, 1]

    Both are empirical averages, not raw sums.
    """
    x_f = x.float()
    y_f = real_points.float()
    m = y_f.shape[1]

    if m < 1:
        raise ValueError("Correction minibatch must be non-empty.")

    weights = _laplacian_kernel_batched(
        x_f,
        y_f,
        temperature=temperature,
        eps=eps,
    )

    numerator = torch.bmm(weights, y_f) / float(m)
    denominator = weights.mean(dim=2, keepdim=True)

    return numerator, denominator


def _compute_nystrom_minibatch_moments_batched(
    phi_x,
    real_points,
    phi_y,
):
    """Nyström moments over the same real-data minibatch.

    Args:
        phi_x: [L, N, M]
        real_points: [L, m, D]
        phi_y: [L, m, M]
    """
    m = real_points.shape[1]

    # Approximate pairwise kernel [L, N, m].
    approx_weights = torch.bmm(
        phi_x.float(),
        phi_y.float().transpose(1, 2),
    )

    numerator = torch.bmm(
        approx_weights,
        real_points.float(),
    ) / float(m)

    denominator = approx_weights.mean(
        dim=2,
        keepdim=True,
    )

    return numerator, denominator


In [6]:
def _extract_spatial_features(feat_map, pool_size=4):
    """Extract per-location + statistical features from a feature map.

    Args:
        feat_map: [B, C, H, W] feature map
        pool_size: target spatial size for adaptive pooling

    Returns:
        List of (features[B, L, C], C_j) tuples.
        Contains: per-location vectors, global mean, global std.
    """
    B, C, H, W = feat_map.shape
    result = []

    if H != pool_size or W != pool_size:
        pooled = F.adaptive_avg_pool2d(feat_map, (pool_size, pool_size))
    else:
        pooled = feat_map

    # Per-location vectors: [B, pool_size^2, C]
    per_loc = pooled.reshape(B, C, pool_size * pool_size).permute(0, 2, 1)
    result.append((per_loc, C))

    # Global mean: [B, 1, C]
    g_mean = feat_map.mean(dim=[2, 3]).unsqueeze(1)
    result.append((g_mean, C))

    # Global std: [B, 1, C]
    g_std = feat_map.reshape(B, C, -1).std(dim=2).unsqueeze(1)
    result.append((g_std, C))

    return result

def _group_by_channel_dim(feature_groups):
    """Group feature tuples by channel dimension for batched computation."""
    by_dim = {}
    for feat, C_j in feature_groups:
        if C_j not in by_dim:
            by_dim[C_j] = []
        by_dim[C_j].append(feat)

    result = []
    for C_j in sorted(by_dim.keys()):
        cat = torch.cat(by_dim[C_j], dim=1)  # [B, sum(L_i), C_j]
        result.append((cat, C_j))
    return result

class DINOv3MultiResEncoder(nn.Module):
    """Frozen DINOv3 ViT-B/16 with multi-resolution patch token extraction via timm.

    Uses timm's forward_intermediates() for clean multi-scale extraction.
    DINOv3 uses RoPE so it handles arbitrary input sizes.
    """

    def __init__(self, pool_size=4, input_size=112):
        super().__init__()
        self.pool_size = pool_size
        self.input_size = input_size
        import timm

        model_name = 'vit_base_patch16_dinov3.lvd1689m'
        self.model = timm.create_model(
            model_name,
            pretrained=True,
            num_classes=0,
        )
        self.model.eval()
        for param in self.model.parameters():
            param.requires_grad = False
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        self.patch_size = 16
        self.hidden_size = 768
        self.layer_indices = [2, 5, 8, 11]  # 0-indexed for timm

    def forward(self, x):
        x = (x + 1) / 2
        x = (x - self.mean) / self.std
        ps = self.patch_size
        target_size = (self.input_size // ps) * ps
        x = F.interpolate(x, size=(target_size, target_size), mode='bilinear', align_corners=False)

        final_feat, intermediates = self.model.forward_intermediates(
            x, indices=self.layer_indices
        )

        all_groups = []
        for feat_map in intermediates:
            if feat_map.dim() == 3:
                h_p = target_size // ps
                feat_map = feat_map.transpose(1, 2).reshape(-1, self.hidden_size, h_p, h_p)
            groups = _extract_spatial_features(feat_map, self.pool_size)
            all_groups.extend(groups)

        return _group_by_channel_dim(all_groups)

def build_encoder(pool_size=4, input_size=112):
        return DINOv3MultiResEncoder(pool_size=pool_size, input_size=input_size)

@torch.no_grad()
def precompute_features(
    encoder,
    dataset,
    device,
    batch_size=256,
    include_horizontal_flip=True,
    storage_dtype=torch.float16,
):
    """Precompute original and horizontally flipped real-image features.

    RandomHorizontalFlip inside the dataset transform would be sampled only
    once because the reference features are cached. Encoding both variants
    explicitly represents the desired augmented empirical distribution.
    """
    print("Pre-computing real image features...")

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
        drop_last=False,
        persistent_workers=True,
    )

    all_groups = None
    all_labels = []
    base_count = 0

    with torch.amp.autocast("cuda", dtype=torch.bfloat16):
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(dtype=torch.long)

            variants = [images]
            if include_horizontal_flip:
                variants.append(torch.flip(images, dims=[3]))

            for variant in variants:
                groups = encoder(variant)

                if all_groups is None:
                    all_groups = [[] for _ in groups]

                for i, (feat, _) in enumerate(groups):
                    all_groups[i].append(
                        feat.detach()
                        .to(dtype=storage_dtype)
                        .cpu()
                    )

                all_labels.append(labels.clone())

            base_count += images.shape[0]
            if base_count % 10_000 == 0:
                print(
                    f"  {base_count}/{len(dataset)} base images"
                )

    features = [
        torch.cat(items, dim=0)
        for items in all_groups
    ]
    cached_labels = torch.cat(all_labels, dim=0)

    print(
        f"  Cached examples: {cached_labels.numel()} "
        f"(horizontal flip={include_horizontal_flip})"
    )

    for i, feat in enumerate(features):
        print(
            f"    Group {i}: shape={list(feat.shape)}, "
            f"dtype={feat.dtype}"
        )

    return features, cached_labels


In [7]:
def format_scientific(value):
    return f"{float(value):.4e}" if float(value) else "0.0000e+00"

def count_feature_slots(groups):
    return sum(int(feat.shape[1]) for feat, _ in groups)

def scaled_loss_for_logging(loss, groups):
    return float(loss.detach().item()) * count_feature_slots(groups)

def append_training_log(path, step, loss, elapsed, images_per_sec):
    with open(path, "a", newline="") as handle:
        csv.writer(handle).writerow([step, loss, elapsed, images_per_sec])

@torch.no_grad()
def select_random_landmarks(
    features,
    count,
    seed,
    device,
):
    """Select landmarks uniformly from one class."""
    count = min(int(count), int(features.shape[0]))

    generator = torch.Generator(
        device="cpu"
    ).manual_seed(int(seed))

    indices = torch.randperm(
        features.shape[0],
        generator=generator,
    )[:count]

    return features.index_select(0, indices).to(
        device=device,
        dtype=torch.float32,
    )


@torch.no_grad()
def build_conditional_nystrom_stats_groups(
    feature_groups,
    labels,
    cfg,
    device,
):
    """Build class-specific attraction caches plus one compact CFG cache."""
    labels = torch.as_tensor(labels, dtype=torch.long)

    class_stats_groups = []
    unconditional_stats_groups = []

    for group_index, features in enumerate(feature_groups):
        print(f"Building Nyström caches for group {group_index}...")

        per_class_stats = {}

        for class_id in range(cfg.num_classes):
            class_indices = torch.nonzero(
                labels == class_id,
                as_tuple=False,
            ).squeeze(1)

            class_features = features.index_select(
                0, class_indices
            )

            landmarks = select_random_landmarks(
                features=class_features,
                count=cfg.nystrom_landmarks_per_class,
                seed=(
                    cfg.nystrom_landmark_seed
                    + class_id * 1009
                    + group_index * 1_000_003
                ),
                device=device,
            )

            per_class_stats[class_id] = (
                precompute_nystrom_statistics_multitemp_batched(
                    sensitive_points=class_features,
                    public_landmarks=landmarks,
                    temps=tuple(cfg.temperatures),
                    ridge=cfg.nystrom_ridge,
                    batch_size=512,
                    device=device,
                )
            )

            print(
                f"  group={group_index}, class={class_id}: "
                f"points={class_features.shape[0]}, "
                f"landmarks={landmarks.shape[0]}"
            )

            del class_features, landmarks

        # Compact unconditional cache used only by CFG.
        unconditional_landmarks = select_random_per_class_landmarks(
            features=features,
            labels=labels,
            landmarks_per_class=(
                cfg.nystrom_uncond_landmarks_per_class
            ),
            seed=cfg.nystrom_landmark_seed + 7_919,
            device=device,
        )

        unconditional_stats = (
            precompute_nystrom_statistics_multitemp_batched(
                sensitive_points=features,
                public_landmarks=unconditional_landmarks,
                temps=tuple(cfg.temperatures),
                ridge=cfg.nystrom_ridge,
                batch_size=512,
                device=device,
            )
        )

        class_stats_groups.append(per_class_stats)
        unconditional_stats_groups.append(unconditional_stats)

        print(
            f"  group={group_index}, unconditional: "
            f"points={features.shape[0]}, "
            f"landmarks={unconditional_landmarks.shape[0]}"
        )

        del unconditional_landmarks
        torch.cuda.empty_cache()

    return class_stats_groups, unconditional_stats_groups

def sample_power_law_cfg(
    count,
    alpha_max,
    power,
    no_guidance_probability,
    device,
):
    """Sample alpha in [1, alpha_max] with density proportional to alpha^-p."""
    count = int(count)
    alpha_max = float(alpha_max)
    power = float(power)

    if alpha_max <= 1.0:
        return torch.ones(count, device=device)

    u = torch.rand(count, device=device)

    if abs(power - 1.0) < 1e-8:
        guided = torch.exp(u * math.log(alpha_max))
    else:
        exponent = 1.0 - power
        guided = (
            1.0
            + u * (alpha_max ** exponent - 1.0)
        ) ** (1.0 / exponent)

    disable_guidance = (
        torch.rand(count, device=device)
        < float(no_guidance_probability)
    )

    return torch.where(
        disable_guidance,
        torch.ones_like(guided),
        guided,
    )


def update_learning_rate(optimizer, step, cfg):
    """Linear warm-up followed by cosine decay."""
    if step <= cfg.warmup_steps:
        lr = cfg.lr * step / max(cfg.warmup_steps, 1)
    else:
        progress = (
            step - cfg.warmup_steps
        ) / max(cfg.total_steps - cfg.warmup_steps, 1)
        progress = min(max(progress, 0.0), 1.0)

        cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
        lr = cfg.min_lr + (cfg.lr - cfg.min_lr) * cosine

    for group in optimizer.param_groups:
        group["lr"] = lr

    return lr

# Fixed random_per_class selection replaces the original generic strategy dispatcher.
# Original source: train_nystrom.py (_sample_random_indices/build_landmarks_for_group).
@torch.no_grad()
def select_random_per_class_landmarks(features, labels, landmarks_per_class, seed, device):
    labels = torch.as_tensor(labels, dtype=torch.long)
    selected = []
    for class_id in torch.unique(labels, sorted=True).tolist():
        indices = torch.nonzero(labels == class_id, as_tuple=False).squeeze(1)
        # Same per-class seed offset as train_nystrom.py's generic selector.
        generator = torch.Generator(device="cpu").manual_seed(int(seed) + int(class_id) * 1009)
        selected.append(indices[torch.randperm(indices.numel(), generator=generator)[:landmarks_per_class]])
    return features.index_select(0, torch.cat(selected)).to(device=device, dtype=torch.float32)

@torch.no_grad()
def build_nystrom_stats_groups(feature_groups, labels, cfg, device):
    stats_groups = []
    for group_index, features in enumerate(feature_groups):
        landmarks = select_random_per_class_landmarks(
            features, labels, cfg.nystrom_landmarks_per_class,
            cfg.nystrom_landmark_seed, device)
        stats_groups.append(precompute_nystrom_statistics_multitemp_batched(
            features, landmarks, temps=tuple(cfg.temperatures), ridge=cfg.nystrom_ridge,
            batch_size=512, device=device))
        print(f"  Group {group_index}: landmarks={landmarks.shape[0]}")
    return stats_groups

def drift_sample(model, n, device, class_labels=None, cfg_scale=None, start_idx=0):
    """Generate CIFAR-10 samples from the drifting model in one forward pass."""
    z = torch.randn(n, 3, 32, 32, device=device)

    num_classes = int(getattr(model, "_eval_num_classes", 0) or 0)
    if class_labels is None and num_classes > 0:
        class_labels = (torch.arange(n, device=device, dtype=torch.long) + start_idx) % num_classes
    if cfg_scale is None:
        cfg_scale = getattr(model, "_eval_cfg_scale", None)

    with torch.no_grad():
        if class_labels is not None or cfg_scale is not None:
            x = model(z, class_labels=class_labels, cfg_scale=cfg_scale)
        else:
            x = model(z)
    return x.clamp(-1, 1)

def save_sample_grid(samples, path, nrow=8):
    """Save a grid of samples to disk. Expects samples in [-1, 1]."""
    grid = make_grid(samples, nrow=nrow, normalize=True, value_range=(-1, 1), padding=2)
    save_image(grid, path)


In [8]:
cfg = MultiResDriftConfig()

if cfg.batch_size != cfg.classes_per_step * cfg.samples_per_class:
    raise ValueError(
        "batch_size must equal classes_per_step * samples_per_class."
    )

data_root = "./data"
out_dir = "outputs/training/nystrom_conditional_cfg_cv_beta0.5"
device = torch.device("cuda:2")

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

encoder_input_size = 112

os.makedirs(out_dir, exist_ok=True)
os.makedirs(os.path.join(out_dir, "samples"), exist_ok=True)
os.makedirs(os.path.join(out_dir, "checkpoints"), exist_ok=True)

# -------------------------------------------------------------------------
# Generator
# -------------------------------------------------------------------------

unet_cfg = UNetConfig()

raw_model = UNet(
    in_ch=unet_cfg.in_ch,
    out_ch=unet_cfg.out_ch,
    base_ch=unet_cfg.base_ch,
    ch_mult=unet_cfg.ch_mult,
    num_res_blocks=unet_cfg.num_res_blocks,
    attn_resolutions=unet_cfg.attn_resolutions,
    dropout=unet_cfg.dropout,
    num_heads=unet_cfg.num_heads,
    num_classes=cfg.num_classes,
).to(memory_format=torch.channels_last).to(device)

n_params = sum(p.numel() for p in raw_model.parameters())
print(f"UNet parameters: {n_params:,}")

# EMA owns an ordinary nn.Module, not a compiled wrapper.
ema = EMA(raw_model, decay=cfg.ema_decay)

# torch.compile wraps the same parameter objects used by raw_model.
model = torch.compile(
    raw_model,
    mode="reduce-overhead",
)

optimizer = torch.optim.AdamW(
    raw_model.parameters(),
    lr=cfg.lr,
    betas=(0.9, 0.999),
    weight_decay=0.0,
    fused=True,
)

# -------------------------------------------------------------------------
# Frozen feature encoder
# -------------------------------------------------------------------------

feat_encoder_raw = build_encoder(
    pool_size=cfg.pool_size,
    input_size=encoder_input_size,
).to(device)

feat_encoder_raw.eval()
feat_encoder = torch.compile(
    feat_encoder_raw,
    mode="reduce-overhead",
)

n_feat_params = sum(
    p.numel() for p in feat_encoder_raw.parameters()
)
print(
    f"Feature encoder: {cfg.encoder}, "
    f"{n_feat_params:,} params (frozen, compiled)"
)
print(
    f"  Input size: {encoder_input_size}x{encoder_input_size}"
)

transform_noflip = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.5, 0.5, 0.5),
        (0.5, 0.5, 0.5),
    ),
])

dataset = datasets.CIFAR10(
    root=data_root,
    train=True,
    download=False,
    transform=transform_noflip,
)

feature_groups, cached_labels = precompute_features(
    encoder=feat_encoder,
    dataset=dataset,
    device=device,
    batch_size=256,
    include_horizontal_flip=cfg.use_horizontal_flip,
    storage_dtype=cfg.feature_cache_dtype,
)

class_stats_groups, unconditional_stats_groups = (
    build_conditional_nystrom_stats_groups(
        feature_groups=feature_groups,
        labels=cached_labels,
        cfg=cfg,
        device=device,
    )
)

cv_banks = None

if cfg.cv_enabled and cfg.cv_samples_per_class > 0:
    cv_banks = build_cv_correction_banks(
        feature_groups=feature_groups,
        labels=cached_labels,
        class_stats_groups=class_stats_groups,
        cfg=cfg,
        device=device,
    )

del feature_groups, cached_labels, dataset
torch.cuda.empty_cache()


UNet parameters: 38,641,027


Feature encoder: dinov3, 85,641,216 params (frozen, compiled)
  Input size: 112x112
Pre-computing real image features...
  50000/50000 base images
  Cached examples: 100000 (horizontal flip=True)
    Group 0: shape=[100000, 72, 768], dtype=torch.float16
Building Nyström caches for group 0...
  group=0, class=0: points=10000, landmarks=328
  group=0, class=1: points=10000, landmarks=328
  group=0, class=2: points=10000, landmarks=328
  group=0, class=3: points=10000, landmarks=328
  group=0, class=4: points=10000, landmarks=328
  group=0, class=5: points=10000, landmarks=328
  group=0, class=6: points=10000, landmarks=328
  group=0, class=7: points=10000, landmarks=328
  group=0, class=8: points=10000, landmarks=328
  group=0, class=9: points=10000, landmarks=328
  group=0, unconditional: points=100000, landmarks=640
Building CV correction bank for group 0...
  group=0, class=0: pool=256, points=27.0 MiB, phi=34.6 MiB
  group=0, class=1: pool=256, points=27.0 MiB, phi=34.6 MiB
  group=0

In [ ]:
# -------------------------------------------------------------------------
# Logging
# -------------------------------------------------------------------------

log_path = os.path.join(out_dir, "loss_log.csv")

with open(log_path, "w", newline="") as handle:
    csv.writer(handle).writerow([
        "step",
        "loss",
        "lr",
        "mean_cfg",
        "time_s",
        "images_per_sec",
    ])

start_time = time.time()
raw_model.train()

# -------------------------------------------------------------------------
# Training
# -------------------------------------------------------------------------

for step in range(1, cfg.total_steps + 1):
    current_lr = update_learning_rate(
        optimizer,
        step,
        cfg,
    )

    # Select only a few classes per iteration so each active class retains
    # a strong exact repulsive set.
    active_classes = torch.randperm(
        cfg.num_classes,
        device=device,
    )[:cfg.classes_per_step]

    class_labels = active_classes.repeat_interleave(
        cfg.samples_per_class
    )

    if cfg.cfg_enabled:
        alpha_per_class = sample_power_law_cfg(
            count=cfg.classes_per_step,
            alpha_max=cfg.cfg_alpha_max,
            power=cfg.cfg_power,
            no_guidance_probability=(
                cfg.cfg_no_guidance_probability
            ),
            device=device,
        )
    else:
        alpha_per_class = torch.ones(
            cfg.classes_per_step,
            device=device,
        )

    cfg_scales = alpha_per_class.repeat_interleave(
        cfg.samples_per_class
    )

    z = torch.randn(
        cfg.batch_size,
        3,
        32,
        32,
        device=device,
    ).to(memory_format=torch.channels_last)

    optimizer.zero_grad(set_to_none=True)

    with torch.amp.autocast(
        "cuda",
        dtype=torch.bfloat16,
    ):
        generated = model(
            z,
            class_labels=class_labels,
            cfg_scale=cfg_scales,
        )

        generated_groups = [
            (feat.float(), channels)
            for feat, channels in feat_encoder(generated.float())
        ]

    if step == 1:
        print(
            "  Loss logging normalized by "
            f"{count_feature_slots(generated_groups)} feature slots"
        )

    loss = drifting_loss_multires_nystrom_conditional(
        gen_groups=generated_groups,
        class_stats_groups=class_stats_groups,
        unconditional_stats_groups=unconditional_stats_groups,
        class_labels=class_labels,
        cfg_scales=cfg_scales,
        cv_banks=cv_banks,
        cfg=cfg,
        temps=tuple(cfg.temperatures),
    )

    # BF16 autocast uses FP32 parameters/gradients and does not require a
    # GradScaler.
    loss.backward()

    grad_norm = torch.nn.utils.clip_grad_norm_(
        raw_model.parameters(),
        cfg.max_grad_norm,
    )

    optimizer.step()
    ema.update(raw_model)

    if not torch.isfinite(loss):
        raise FloatingPointError(
            f"Non-finite loss encountered at step {step}: {loss.item()}"
        )

    if step % cfg.log_every == 0:
        logged_loss = scaled_loss_for_logging(
            loss,
            generated_groups,
        )
        elapsed = time.time() - start_time
        images_per_sec = step * cfg.batch_size / elapsed
        iterations_per_sec = step / elapsed
        mean_cfg = float(cfg_scales.mean().item())

        print(
            f"step {step:>7d}/{cfg.total_steps} | "
            f"loss {format_scientific(logged_loss)} | "
            f"lr {current_lr:.3e} | "
            f"cfg {mean_cfg:.2f} | "
            f"grad {float(grad_norm):.2f} | "
            f"{iterations_per_sec:.2f} it/s "
            f"({images_per_sec:.0f} img/s) | "
            f"ETA "
            f"{(cfg.total_steps-step)/iterations_per_sec/3600:.1f}h"
        )

        with open(log_path, "a", newline="") as handle:
            csv.writer(handle).writerow([
                step,
                format_scientific(logged_loss),
                f"{current_lr:.8e}",
                f"{mean_cfg:.4f}",
                f"{elapsed:.1f}",
                f"{images_per_sec:.0f}",
            ])

    if step % cfg.sample_every == 0:
        raw_model.eval()

        # Eight samples per CIFAR-10 class, ordered by class.
        samples_per_eval_class = 8
        sample_labels = torch.arange(
            cfg.num_classes,
            device=device,
            dtype=torch.long,
        ).repeat_interleave(samples_per_eval_class)

        sample_cfg = torch.full(
            (sample_labels.numel(),),
            float(cfg.eval_cfg_scale),
            device=device,
        )

        samples = drift_sample(
            ema.shadow,
            n=sample_labels.numel(),
            device=device,
            class_labels=sample_labels,
            cfg_scale=sample_cfg,
        )

        save_sample_grid(
            samples,
            os.path.join(
                out_dir,
                "samples",
                f"drift_step{step:07d}_cfg"
                f"{cfg.eval_cfg_scale:.2f}.png",
            ),
            nrow=samples_per_eval_class,
        )

        raw_model.train()
        print(f"  Saved class-conditional grid at step {step}")

    if step % cfg.save_every == 0:
        checkpoint = {
            "step": step,
            "model": raw_model.state_dict(),
            "ema": ema.state_dict(),
            "optimizer": optimizer.state_dict(),
            "config": {
                "unet": unet_cfg,
                "drift": cfg,
            },
        }

        checkpoint_path = os.path.join(
            out_dir,
            "checkpoints",
            f"drift_step{step:07d}.pt",
        )

        torch.save(checkpoint, checkpoint_path)
        torch.save(
            checkpoint,
            os.path.join(
                out_dir,
                "checkpoints",
                "drift_latest.pt",
            ),
        )

        print(f"  Saved checkpoint at step {step}")

elapsed = time.time() - start_time

print(
    f"\nTraining complete. {cfg.total_steps} steps in "
    f"{elapsed / 3600:.1f} hours"
)

torch.save(
    {
        "step": cfg.total_steps,
        "model": raw_model.state_dict(),
        "ema": ema.state_dict(),
        "config": {
            "unet": unet_cfg,
            "drift": cfg,
        },
    },
    os.path.join(
        out_dir,
        "checkpoints",
        "drift_final.pt",
    ),
)


  Loss logging normalized by 72 feature slots
step     100/50000 | loss 5.8171e+02 | lr 1.000e-05 | cfg 2.20 | grad 1914.15 | 2.94 it/s (753 img/s) | ETA 4.7h
step     200/50000 | loss 5.8159e+02 | lr 2.000e-05 | cfg 1.00 | grad 1695.47 | 3.56 it/s (911 img/s) | ETA 3.9h
step     300/50000 | loss 5.7057e+02 | lr 3.000e-05 | cfg 1.58 | grad 1211.05 | 3.81 it/s (974 img/s) | ETA 3.6h
step     400/50000 | loss 5.6975e+02 | lr 4.000e-05 | cfg 1.46 | grad 367.99 | 3.93 it/s (1005 img/s) | ETA 3.5h
step     500/50000 | loss 5.5974e+02 | lr 5.000e-05 | cfg 1.99 | grad 199.89 | 3.99 it/s (1022 img/s) | ETA 3.4h
step     600/50000 | loss 5.6775e+02 | lr 6.000e-05 | cfg 1.13 | grad 221.53 | 4.04 it/s (1033 img/s) | ETA 3.4h
step     700/50000 | loss 5.6094e+02 | lr 7.000e-05 | cfg 1.62 | grad 136.93 | 4.07 it/s (1041 img/s) | ETA 3.4h
step     800/50000 | loss 5.5715e+02 | lr 8.000e-05 | cfg 1.27 | grad 60.00 | 4.09 it/s (1046 img/s) | ETA 3.3h
step     900/50000 | loss 5.5220e+02 | lr 9.000e-05